# Henon-Heiles System

This notebook trains two neural networks for the Henon-Heiles system and compares them with an RK4 trajectory.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json
import os
import sys

import matplotlib
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm

matplotlib.rcParams["font.family"] = "Times New Roman"
matplotlib.rcParams["font.size"] = 11
matplotlib.rcParams["mathtext.fontset"] = "stix"
matplotlib.rcParams["axes.linewidth"] = 0.8
matplotlib.rcParams["xtick.major.width"] = 0.8
matplotlib.rcParams["ytick.major.width"] = 0.8

cwd = Path.cwd()
EXAMPLES_DIR = cwd if cwd.name == "examples" else cwd / "examples"
PROJECT_ROOT = EXAMPLES_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    DATA_TYPE,
    DEVICE,
    SimpleMLP,
    ensure_run_dirs,
    expected_path_tensor,
    second_order_dynamics,
)

print(f"Using device: {DEVICE}")

In [ ]:
@dataclass(frozen=True)
class HenonHeiles:
    dim: int = 2

    def acceleration(self, position: torch.Tensor) -> torch.Tensor:
        x = position[..., 0]
        y = position[..., 1]
        ax = -x - 2.0 * x * y
        ay = -y - x**2 + y**2
        return torch.stack([ax, ay], dim=-1)


def plot_trajectories(time_values, predicted, reference):
    fig = plt.figure(figsize=(12, 5), dpi=150)
    fig.patch.set_facecolor("white")
    gs = fig.add_gridspec(2, 2, height_ratios=[0.1, 1], hspace=0.3, wspace=0.3, top=0.92)

    ax_legend = fig.add_subplot(gs[0, :])
    ax_legend.axis("off")
    line_neural, = ax_legend.plot([], [], color="black", linewidth=2, label="Neural IVP")
    line_reference, = ax_legend.plot([], [], color="lightgrey", linewidth=2.5, label="RK4")
    ax_legend.legend(handles=[line_neural, line_reference], loc="upper center", ncol=2, frameon=False)

    labels = [r"$x(t)$", r"$y(t)$"]
    for index, label in enumerate(labels):
        ax = fig.add_subplot(gs[1, index])
        ax.plot(time_values, predicted[:, index], color="black", linewidth=2)
        ax.plot(time_values, reference[:, index], color="lightgrey", linewidth=2.5, zorder=-1)
        ax.set_xlabel(r"$t$")
        ax.set_ylabel(label)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(direction="in", which="both", left=True, bottom=True)
        ax.tick_params(labelbottom=True, labelleft=True)
    return fig


def plot_phase_space(predicted, reference):
    fig = plt.figure(figsize=(6, 6), dpi=150)
    fig.patch.set_facecolor("white")
    ax = fig.add_subplot(111)
    ax.plot(predicted[:, 0], predicted[:, 1], color="black", linewidth=2, label="Neural IVP")
    ax.plot(reference[:, 0], reference[:, 1], color="lightgrey", linewidth=2.5, label="RK4", zorder=-1)
    ax.set_xlabel(r"$x(t)$")
    ax.set_ylabel(r"$y(t)$")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="in", which="both", left=True, bottom=True)
    ax.tick_params(labelbottom=True, labelleft=True)
    return fig


def plot_loss_history(loss_history):
    fig = plt.figure(figsize=(8, 5), dpi=150)
    fig.patch.set_facecolor("white")
    ax = fig.add_subplot(111)
    ax.semilogy(loss_history, linewidth=2, color="black")
    ax.set_xlabel(r"Epoch")
    ax.set_ylabel(r"Loss")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="in", which="both", left=True, bottom=True)
    ax.tick_params(labelbottom=True, labelleft=True)
    return fig

In [ ]:
scenarios = {
    "chaotic": {
        "activation": "sechlu",
        "hidden": 64,
        "epochs": 100_000,
        "lr": 3e-3,
        "T_MIN": 0.0,
        "T_MAX": 15.0,
        "TIMESTEP": 0.05,
        "ENERGY": 1 / 6,
        "x0": 0.0,
        "y0": 0.57,
        "vx0": 0.10,
        "vy0": 0.057,
        "ODE_REGULARISER": 1.0,
        "POSITION_REGULARISER": 1.0,
        "VELOCITY_REGULARISER": 1.0,
        "JOB_ID": 1,
    },
    "quasiperiodic": {
        "activation": "sechlu",
        "hidden": 64,
        "epochs": 200,
        "lr": 3e-3,
        "T_MIN": 0.0,
        "T_MAX": 15.0,
        "TIMESTEP": 0.01,
        "ENERGY": 1 / 12,
        "x0": 0.0,
        "y0": 0.40,
        "vx0": 0.05,
        "vy0": 0.041,
        "ODE_REGULARISER": 1.0,
        "POSITION_REGULARISER": 1.0,
        "VELOCITY_REGULARISER": 1.0,
        "JOB_ID": 2,
    },
}

selected_scenario = "chaotic"
params = scenarios[selected_scenario]

dt = params["TIMESTEP"]
steps = int(round((params["T_MAX"] - params["T_MIN"]) / dt)) + 1
time_grid = torch.linspace(
    params["T_MIN"],
    params["T_MAX"],
    steps,
    dtype=DATA_TYPE,
    device=DEVICE,
)
t = time_grid.unsqueeze(0).unsqueeze(2).requires_grad_(True)

init_position = torch.tensor([params["x0"], params["y0"]], dtype=DATA_TYPE, device=DEVICE)
init_velocity = torch.tensor([params["vx0"], params["vy0"]], dtype=DATA_TYPE, device=DEVICE)
system = HenonHeiles()
expected = expected_path_tensor(
    system,
    init_position,
    init_velocity,
    params["T_MIN"],
    params["T_MAX"],
    dt,
)

In [ ]:
models = [
    SimpleMLP(1, params["hidden"], 1, params["activation"]).to(DEVICE),
    SimpleMLP(1, params["hidden"], 1, params["activation"]).to(DEVICE),
]
optimisers = [torch.optim.Adam(model.parameters(), lr=params["lr"]) for model in models]
loss_fn = torch.nn.MSELoss()
loss_history = []


def evaluate():
    coordinates = [model(t) for model in models]
    positions = torch.cat(coordinates, dim=2)
    velocities, _, residuals = second_order_dynamics(system, positions, t)
    position_loss = loss_fn(positions[:, 0, :], init_position.unsqueeze(0))
    velocity_loss = loss_fn(velocities[:, 0, :], init_velocity.unsqueeze(0))
    ode_loss = loss_fn(residuals, torch.zeros_like(residuals))
    total_loss = (
        params["ODE_REGULARISER"] * ode_loss
        + params["POSITION_REGULARISER"] * position_loss
        + params["VELOCITY_REGULARISER"] * velocity_loss
    )
    return positions, total_loss


for epoch in tqdm(range(params["epochs"])):
    positions, total_loss = evaluate()
    for optimiser in optimisers:
        optimiser.zero_grad()
    total_loss.backward()
    for optimiser in optimisers:
        optimiser.step()
    loss_history.append(total_loss.item())

    if epoch % 500 == 0 or epoch == params["epochs"] - 1:
        time_values = t[0, :, 0].detach().cpu().numpy()
        predicted_values = positions[0].detach().cpu().numpy()
        reference_values = expected[0].detach().cpu().numpy()
        display(plot_trajectories(time_values, predicted_values, reference_values))
        display(plot_phase_space(predicted_values, reference_values))
        display(plot_loss_history(loss_history))

final_positions, _ = evaluate()

In [ ]:
output_base = EXAMPLES_DIR / "outputs"
paths = ensure_run_dirs(str(output_base), params, system_name="henon_heiles")
model_x_path = os.path.join(paths["models"], f"{params['JOB_ID']}_x.pth")
model_y_path = os.path.join(paths["models"], f"{params['JOB_ID']}_y.pth")
loss_history_path = os.path.join(paths["paths"], "loss_history.json")
trajectories_path = os.path.join(paths["figures"], "trajectories.png")
phase_space_path = os.path.join(paths["figures"], "phase_space.png")
loss_plot_path = os.path.join(paths["figures"], "loss_history.png")

torch.save(models[0], model_x_path)
torch.save(models[1], model_y_path)
with open(loss_history_path, "w") as handle:
    json.dump(loss_history, handle)

time_values = t[0, :, 0].detach().cpu().numpy()
predicted_values = final_positions[0].detach().cpu().numpy()
reference_values = expected[0].detach().cpu().numpy()

trajectories_figure = plot_trajectories(time_values, predicted_values, reference_values)
trajectories_figure.savefig(trajectories_path, dpi=150, bbox_inches="tight")
phase_figure = plot_phase_space(predicted_values, reference_values)
phase_figure.savefig(phase_space_path, dpi=150, bbox_inches="tight")
loss_figure = plot_loss_history(loss_history)
loss_figure.savefig(loss_plot_path, dpi=150, bbox_inches="tight")

print(f"Scenario: {selected_scenario}")
print("Saved:")
print(f"- Model X: {model_x_path}")
print(f"- Model Y: {model_y_path}")
print(f"- Loss history: {loss_history_path}")
print(f"- Trajectories: {trajectories_path}")
print(f"- Phase space: {phase_space_path}")
print(f"- Loss plot: {loss_plot_path}")